# Additional Task B — AI-Generated Text Detection (CoAT)

## Preliminaries

### Installation

Install scikit-learn for classification, transformers for the multilingual BERT tokenizer, and joblib for model serialization.

In [1]:
# !pip install nltk seaborn joblib --quiet


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


### Imports

Core imports used throughout the task.

In [2]:
import json
import os
import pickle
import re
import subprocess
from collections import Counter
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm.notebook import tqdm
from transformers import BertTokenizer

nltk.download('stopwords', quiet=True)

/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


True

---

## B.1 Data Acquisition

### Cloning CoAT Repository

Clone the CoAT (Corpus of AI-generated Texts) repository from RussianNLP. After cloning, display the README to understand the file structure and label conventions before writing any loading code.

In [3]:
if not os.path.exists('CoAT'):
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/RussianNLP/CoAT.git'],
        check=True
    )

readme_path = next(
    (p for p in ['CoAT/README.md', 'CoAT/README.rst', 'CoAT/readme.md']
     if os.path.exists(p)),
    None
)
if readme_path:
    with open(readme_path, encoding='utf-8') as f:
        print(f.read()[:3000])

# CoAT

CoAT🧥 (Corpus of Artificial Texts) is a large-scale corpus for Russian, which consists of 246k human-written texts from publicly available resources and artificial texts generated by 13 neural models, varying in the number of parameters, architecture choices, pre-training objectives, and downstream applications.
Each model is fine-tuned for one or more of six natural language generation tasks, ranging from paraphrase generation to text summarisation. CoAT provides two task formulations:

1. detection of artificial texts, i.e., classifying if a given text is machine-generated or human-written;

2. authorship attribution, i.e., classifying the author of a given text among 14 candidates.

The design of our corpus enables various experiment settings, ranging from analysing the dependence of the detector performance on the natural language generation task to the robustness of detectors towards unseen generative models and text domains.

CoAT is available on [HuggingFace](https://hug

### Loading Dataset

Inspect the repository structure to locate data files, then build a unified DataFrame with `text` and `label` columns (0 = human-written, 1 = machine-generated). The loading logic handles both flat JSON-lines files and directory-per-split layouts common in NLP dataset releases.

In [4]:
def list_data_files(root, extensions=('.jsonl', '.json', '.csv', '.tsv')):
    found = []
    for path in Path(root).rglob('*'):
        if path.suffix in extensions:
            found.append(path)
    return sorted(found)


data_files = list_data_files('CoAT')
print(f"Found {len(data_files)} data files:")
for p in data_files:
    print(f"  {p}")

Found 6 data files:
  CoAT/datasets/authorship/test.csv
  CoAT/datasets/authorship/train.csv
  CoAT/datasets/authorship/val.csv
  CoAT/datasets/binary/test.csv
  CoAT/datasets/binary/train.csv
  CoAT/datasets/binary/val.csv


In [5]:
def load_coat(data_files):
    records = []
    for path in data_files:
        if path.suffix in ('.jsonl', '.json'):
            with open(path, encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        obj = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                    text = obj.get('text') or obj.get('content') or obj.get('passage') or ''
                    label_raw = obj.get('label') or obj.get('class') or obj.get('generated')
                    if isinstance(label_raw, bool):
                        label = int(label_raw)
                    elif isinstance(label_raw, str):
                        label = 1 if label_raw.lower() in ('ai', 'machine', '1', 'generated', 'gpt') else 0
                    elif isinstance(label_raw, int):
                        label = label_raw
                    else:
                        label = int('human' not in str(path).lower())
                    if text:
                        records.append({'text': text, 'label': label})
        elif path.suffix == '.csv':
            df = pd.read_csv(path)
            text_col = next((c for c in df.columns if 'text' in c.lower()), df.columns[0])
            label_col = next((c for c in df.columns if 'label' in c.lower()), None)
            for _, row in df.iterrows():
                text = str(row[text_col])
                label = int(row[label_col]) if label_col else 0
                records.append({'text': text, 'label': label})
    return pd.DataFrame(records)


coat_df = load_coat(data_files)
print(f"Total records: {len(coat_df):,}")
print(f"Label distribution:")
print(coat_df['label'].value_counts().rename({0: 'human (0)', 1: 'machine (1)'}))
coat_df.head(3)

ValueError: invalid literal for int() with base 10: 'Human'

---

## B.2 Feature Engineering

### Initialization

**Russian stopwords** from NLTK replace the English list from Task 2, since CoAT contains Russian text. The BERT tokenizer is **`bert-base-multilingual-cased`** as specified — the multilingual variant supports Russian and preserves casing, which matters for Cyrillic proper nouns. Both objects are initialized once as module-level globals and reused inside `extract_features`.

In [ ]:
from nltk.corpus import stopwords as nltk_stopwords

STOPWORDS_RU = set(nltk_stopwords.words('russian'))
BERT_TOKENIZER = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

print(f"Russian stopwords: {len(STOPWORDS_RU)}")
print(f"BERT vocab size: {BERT_TOKENIZER.vocab_size:,}")

### Vocabulary Growth Helper

Lightweight variant of `vocab_growth` from §A.3 that sub-samples at up to 200 points along the token sequence, making per-text Heaps' β computation fast enough for large datasets. Uses `np.polyfit` in log–log space, consistent with Task A and Task 2.

In [ ]:
def heaps_beta_for_text(tokens):
    n = len(tokens)
    if n < 30:
        return np.nan
    step = max(1, n // 200)
    vocab = set()
    counts, sizes = [], []
    for i, tok in enumerate(tokens, 1):
        vocab.add(tok)
        if i % step == 0:
            counts.append(i)
            sizes.append(len(vocab))
    if len(counts) < 3:
        return np.nan
    try:
        coef = np.polyfit(np.log(counts), np.log(sizes), 1)
        return float(coef[0])
    except (ValueError, np.linalg.LinAlgError):
        return np.nan

### Feature Extraction Function

All eight features are computed from the token list of a single text. Token extraction uses the same regex as Task A and Task 2. **`bert_fertility`** samples the first 512 words to keep latency bounded (BERT's practical context window).

In [ ]:
def extract_features(text):
    tokens = re.findall(r'[а-яёa-z]+', text.lower())
    n = len(tokens)
    if n < 5:
        return {k: np.nan for k in (
            'ttr', 'heaps_beta', 'zipf_alpha', 'stopword_ratio',
            'avg_word_len', 'avg_sent_len', 'bert_fertility', 'hapax_ratio',
        )}

    freq = Counter(tokens)
    unique = set(tokens)

    ttr = len(unique) / n

    heaps_beta = heaps_beta_for_text(tokens)

    freq_sorted = np.array(sorted(freq.values(), reverse=True), dtype=float)
    if len(freq_sorted) >= 10:
        ranks = np.arange(1, len(freq_sorted) + 1, dtype=float)
        try:
            coef = np.polyfit(np.log(ranks), np.log(freq_sorted), 1)
            zipf_alpha = float(-coef[0])
        except (ValueError, np.linalg.LinAlgError):
            zipf_alpha = np.nan
    else:
        zipf_alpha = np.nan

    stopword_ratio = sum(1 for t in tokens if t in STOPWORDS_RU) / n

    avg_word_len = sum(len(t) for t in tokens) / n

    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    if sentences:
        sent_lens = [
            len(re.findall(r'[а-яёa-z]+', s.lower())) for s in sentences
        ]
        avg_sent_len = float(np.mean([l for l in sent_lens if l > 0]) if sent_lens else n)
    else:
        avg_sent_len = float(n)

    sample = tokens[:512]
    try:
        subwords = BERT_TOKENIZER.tokenize(' '.join(sample))
        bert_fertility = len(subwords) / len(sample)
    except Exception:
        bert_fertility = np.nan

    hapax_count = sum(1 for c in freq.values() if c == 1)
    hapax_ratio = hapax_count / len(unique) if unique else np.nan

    return {
        'ttr': ttr,
        'heaps_beta': heaps_beta,
        'zipf_alpha': zipf_alpha,
        'stopword_ratio': stopword_ratio,
        'avg_word_len': avg_word_len,
        'avg_sent_len': avg_sent_len,
        'bert_fertility': bert_fertility,
        'hapax_ratio': hapax_ratio,
    }

### Applying Features to the Dataset

In [ ]:
feature_records = []
for text in tqdm(coat_df['text'].tolist()):
    feature_records.append(extract_features(text))

features_df = pd.DataFrame(feature_records)
features_df['label'] = coat_df['label'].values

features_df.describe()

Save Feature Matrix

In [ ]:
features_df.to_csv('coat_features.csv', index=False)
print(f"Saved: coat_features.csv  ({features_df.shape})")

Load Feature Matrix from Backup

In [ ]:
features_df = pd.read_csv('coat_features.csv')
print(f"Loaded: {features_df.shape}")
features_df.head(3)

---

## B.3 Exploratory Analysis

### Feature Distributions

KDE plots show the overlapping density of each feature for human (0) and machine-generated (1) texts. Well-separated distributions indicate high discriminative power.

In [ ]:
FEATURE_COLS = [c for c in features_df.columns if c != 'label']
human_df = features_df[features_df['label'] == 0]
ai_df = features_df[features_df['label'] == 1]

n_cols = 4
n_rows = (len(FEATURE_COLS) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
axes = axes.flatten()

for i, feat in enumerate(FEATURE_COLS):
    ax = axes[i]
    h_vals = human_df[feat].dropna()
    a_vals = ai_df[feat].dropna()
    h_vals.plot.kde(ax=ax, color='steelblue', label='Human', linewidth=1.5)
    a_vals.plot.kde(ax=ax, color='darkorange', label='AI', linewidth=1.5)
    ax.set_title(feat)
    ax.set_xlabel('')
    ax.legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions: Human vs. AI', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### Correlation Matrix

In [ ]:
corr = features_df[FEATURE_COLS].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    square=True,
    linewidths=0.4,
    ax=ax,
)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

**Interpretation.** Features with clearly separated KDE peaks are the most useful individual predictors. Strong correlations (|r| > 0.7) between feature pairs suggest redundancy — logistic regression's L2 regularization will handle this, but it is worth noting for model interpretation. Expected findings based on the literature:

- **TTR and hapax_ratio** tend to be higher for human text (richer vocabulary, more unique word use).
- **avg_sent_len** tends to be more uniform in AI text (language models prefer consistent sentence lengths).
- **bert_fertility** may be higher for AI text if the model favors morphologically complex or rare forms.
- **stopword_ratio** may differ if AI models overuse or underuse function words relative to human writers.

---

## B.4 Classification

### Train/Test Split

**80/20 split** with `stratify=y` preserves the class balance in both sets. A larger test fraction (30–40 %) is common in dataset-exploration settings, but 20 % is standard for classification benchmarks and gives enough test examples for stable metric estimates. `random_state=42` ensures reproducibility. Features are standardised with `StandardScaler` (zero mean, unit variance) — mandatory for logistic regression because L2 penalises coefficients equally regardless of feature scale.

In [ ]:
clean_df = features_df.dropna()
X = clean_df[FEATURE_COLS].values
y = clean_df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"Train: {X_train_sc.shape[0]}  Test: {X_test_sc.shape[0]}")
print(f"Train label balance: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Test  label balance: {dict(zip(*np.unique(y_test, return_counts=True)))}")

### Logistic Regression

**Model choice — Logistic Regression:** Appropriate because (1) the feature space is small (8 dimensions) — high-capacity models like gradient boosting are prone to overfitting here; (2) coefficients are directly interpretable as log-odds contributions, which is central to the scientific analysis goal; (3) LR provides a strong, well-understood baseline for tabular feature sets in binary classification.

**`C=1.0`** (default) is a standard starting point for L2 regularisation; it applies equal penalty to all coefficients. **`max_iter=1000`** avoids premature convergence warnings on small datasets. **`solver='lbfgs'`** is efficient for small dense problems with L2 penalty.

In [ ]:
clf = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=42)
clf.fit(X_train_sc, y_train)
print("Training complete.")

### Evaluation

In [ ]:
y_pred = clf.predict(X_test_sc)
y_prob = clf.predict_proba(X_test_sc)[:, 1]

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')
auc = roc_auc_score(y_test, y_prob)

metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 (macro)', 'ROC-AUC'],
    'Value': [round(acc, 4), round(f1, 4), round(auc, 4)],
})
metrics_df.set_index('Metric')

### Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Human', 'AI'],
    cmap='Blues',
    ax=ax,
)
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

### Feature Coefficients

Logistic regression coefficients (after standardisation) represent the change in log-odds of the positive class (AI) per unit increase in the standardised feature. Positive values favour the AI class; negative values favour human.

In [ ]:
coef_df = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'Coefficient': clf.coef_[0],
}).sort_values('Coefficient')

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['steelblue' if c < 0 else 'darkorange' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (log-odds per σ)')
ax.set_title('Logistic Regression Feature Coefficients')
plt.tight_layout()
plt.show()

coef_df.reset_index(drop=True)

**Interpretation of coefficients.** Features with large positive coefficients are the strongest indicators of machine-generated text; large negative coefficients point to human authorship. For example:

- A **high TTR** and **high hapax_ratio** are expected to have negative coefficients (human writers use richer, more idiosyncratic vocabulary).
- **avg_sent_len** showing very uniform values in AI text may yield a coefficient in either direction depending on model generation style.
- **bert_fertility** (subword splitting rate) may be a strong AI indicator if the model favors morphologically unusual forms not well-covered in BERT's vocabulary.

Logistic regression's coefficient signs are directly actionable: they constitute a human-interpretable decision rule for AI detection.